# Raw Data Analysis

Purpose: examines whether property-indixed contingency learning transfers between tasks. 

Data Source: behavioural experiment run on Prolific (n=80)

Date: Friday 3rd April, 2026

This notebook loads every raw CSV in data/raw, concatenates them, applies basic EDA, and applies an initial filter the pre-registered inclusion / exclusion criteria. Frequentist tests and model fitting analysis will be conducted on 'XXX name of notebook' and "XXX name of notebook 2".

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

## Load & Clean

In [2]:
#1. Load and concatenation

data_dir = Path("/Users/ninaedgley/contingency_simulation/data/raw")

dfs = []
excluded = []

for f in data_dir.glob("*.csv"):
    df = pd.read_csv(f)
    is_test = f.name.startswith("test_")
    wrong_length = len(df) != 728

    if is_test or wrong_length:
        excluded.append((f.name, len(df)))
        continue

    debrief = df[df['phase'] == 'debrief']
    completion_ms = debrief['time_elapsed'].values[0] if len(debrief) > 0 else np.nan
    df['completion_ms'] = completion_ms
    
    dfs.append(df)

raw = pd.concat(dfs, ignore_index=True)

print(f'Included files: {len(dfs)}')
print(f'Excluded files: {len(excluded)}')
print(f'Total rows: {len(raw)}')
print(f'Subjects: {raw["subject_id"].nunique()}')


# 2. Outcome phase rows filtered out

df = raw[raw['phase'] == 'outcome'].copy()
print(f'Outcome rows: {len(df)}')


# 3. Drop uninformative + Prolific-native rows
drop_cols = ['stimulus', 'response', 'trial_type', 'trial_index', 'time_elapsed', 'internal_node_id']
df = df.drop(columns=drop_cols)


# 4. Cast types
df['pressed'] = pd.to_numeric(df['pressed'], errors='coerce').astype('Int64')
df['outcome'] = pd.to_numeric(df['outcome'], errors='coerce').astype('Int64')
df['outcome_shown'] = pd.to_numeric(df['outcome_shown'], errors='coerce').astype('Int64')
df['watch_only'] = pd.to_numeric(df['watch_only'], errors='coerce').astype('Int64')
df['rt'] = pd.to_numeric(df['rt'], errors='coerce')  # null → NaN, not 0
df['task'] = pd.to_numeric(df['task'], errors='coerce').astype('Int64')
df['block'] = pd.to_numeric(df['block'], errors='coerce').astype('Int64')
df['trial_n'] = pd.to_numeric(df['trial_n'], errors='coerce').astype('Int64')

Included files: 74
Excluded files: 23
Total rows: 53872
Subjects: 74
Outcome rows: 13320


## Pre-Registered Exclusions
OSF pre-registration : "completion time under 10 minutes (insufficient engagement); overall press rate below 5% or above 95% across all trials (non-engagement or perseveration); more than 20% of trials with missing press data. [...] Outlier subjects: press rates more than 3 SD from the group mean on any primary measure will be flagged and reported but not automatically excluded — sensitivity analyses will be run with and without these subjects."

In [3]:
# A - Completion time < 10 minutes
df['completion_min'] = df['completion_ms'] / 60000
exclude_time = df.groupby('subject_id')['completion_min'].first()
exclude_time = exclude_time[exclude_time < 10].index
print(f'Excluded (completion < 10min): {len(exclude_time)}')

# B - Press rate < 5% or > 95%
press_rate = df[df['watch_only'] == 0].groupby('subject_id')['pressed'].mean()
exclude_press = press_rate[(press_rate < 0.05) | (press_rate > 0.95)].index
print(f'Excluded (press rate floor/ceiling): {len(exclude_press)}')

# C - Missing trial data (> 20% outcome rows missing pressed value)
missing_rate = df.groupby('subject_id')['pressed'].apply(lambda x: x.isna().mean())
exclude_missing = missing_rate[missing_rate > 0.2].index
print(f'Excluded (missing data): {len(exclude_missing)}')


# Combine exclusions
all_excluded = set(exclude_time) | set(exclude_press) | set(exclude_missing)
print(f'Total excluded subjects: {len(all_excluded)}')
print(f'Remaining subjects: {df["subject_id"].nunique() - len(all_excluded)}')

df_clean = df[~df['subject_id'].isin(all_excluded)].copy()
df_clean = df_clean.sort_values(['subject_id', 'task', 'block', 'trial_n'])

df_clean.to_csv('/Users/ninaedgley/contingency_simulation/data/processed/clean_data.csv', index=False)
print(f'\nClean data saved: {len(df_clean)} rows, {df_clean["subject_id"].nunique()} subjects')
print(f'Columns retained: {list(df_clean.columns)}')

Excluded (completion < 10min): 0
Excluded (press rate floor/ceiling): 4
Excluded (missing data): 0
Total excluded subjects: 4
Remaining subjects: 70

Clean data saved: 12600 rows, 70 subjects
Columns retained: ['rt', 'subject_id', 'group', 'phase', 'task', 'block', 'trial_n', 'watch_only', 'prop_SC', 'prop_OC', 'prop_R', 'trial_colour', 'trial_size', 'trial_velocity', 'trial_role', 'pressed', 'outcome', 'outcome_shown', 'completion_ms', 'completion_min']


In [4]:
# ── Sanity Checks ──────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/clean_data.csv')
trials = df[df['watch_only'] == 0].copy()
trials['pressed'] = pd.to_numeric(trials['pressed'], errors='coerce')
trials['outcome'] = pd.to_numeric(trials['outcome'], errors='coerce')

print("=" * 60)
print("SANITY CHECK 1: Outcome probabilities vs. design")
print("=" * 60)
# Expected: SC+pressed≈0.95, SC+notpressed≈0.05, OC≈0.90, R≈0.10
outcome_rates = trials.groupby(['trial_role', 'pressed'])['outcome'].agg(
    ['mean', 'count']
).round(3)
print(outcome_rates)
print()
print("Expected:")
print("  SC, pressed=1 → ~0.95")
print("  SC, pressed=0 → ~0.05")
print("  OC (any)      → ~0.90")
print("  R  (any)      → ~0.10")

print()
print("=" * 60)
print("SANITY CHECK 2: OC/R property swap between tasks")
print("=" * 60)
# For each subject, OC in task 1 should be R in task 2 and vice versa
swap_check = df.groupby(['subject_id', 'task'])[['prop_OC', 'prop_R']].first()
swap_check = swap_check.unstack('task')
swap_check.columns = ['OC_task1', 'OC_task2', 'R_task1', 'R_task2']

# Verify: OC_task1 == R_task2 and R_task1 == OC_task2
swap_check['oc_r_swapped'] = swap_check['OC_task1'] == swap_check['R_task2']
swap_check['r_oc_swapped'] = swap_check['R_task1'] == swap_check['OC_task2']
swap_check['swap_correct'] = swap_check['oc_r_swapped'] & swap_check['r_oc_swapped']

n_correct = swap_check['swap_correct'].sum()
n_total = len(swap_check)
print(f"Subjects with correct OC/R swap: {n_correct}/{n_total}")
if n_correct < n_total:
    print("WARNING: Some subjects did not have correct swap:")
    print(swap_check[~swap_check['swap_correct']])

print()
print("=" * 60)
print("SANITY CHECK 3: Colour values by task")
print("=" * 60)
# Task 1 should only show red/blue. Task 2 should only show green/purple.
colour_by_task = df.groupby(['subject_id', 'task'])['trial_colour'].unique()
colour_check = colour_by_task.reset_index()
colour_check['task1_correct'] = colour_check.apply(
    lambda r: set(r['trial_colour']).issubset({'blue','red'}) 
    if r['task'] == 1 else True, axis=1
)
colour_check['task2_correct'] = colour_check.apply(
    lambda r: set(r['trial_colour']).issubset({'green','purple'}) 
    if r['task'] == 2 else True, axis=1
)
n_t1_correct = colour_check[colour_check['task'] == 1]['task1_correct'].sum()
n_t2_correct = colour_check[colour_check['task'] == 2]['task2_correct'].sum()
n_subjects = df['subject_id'].nunique()
print(f"Task 1 colour values correct (red/blue only): {n_t1_correct}/{n_subjects}")
print(f"Task 2 colour values correct (green/purple only): {n_t2_correct}/{n_subjects}")

print()
print("=" * 60)
print("SANITY CHECK 4: SC property constant across tasks")
print("=" * 60)
sc_by_task = df.groupby(['subject_id', 'task'])['prop_SC'].first().unstack('task')
sc_by_task.columns = ['SC_task1', 'SC_task2']
sc_consistent = (sc_by_task['SC_task1'] == sc_by_task['SC_task2']).all()
print(f"SC property identical across tasks for all subjects: {sc_consistent}")
if not sc_consistent:
    print("WARNING: SC property differs between tasks for some subjects:")
    print(sc_by_task[sc_by_task['SC_task1'] != sc_by_task['SC_task2']])

print()
print("=" * 60)
print("SANITY CHECK 5: Group balance and SC property distribution")
print("=" * 60)
subject_props = df.groupby('subject_id').agg(
    group=('group', 'first'),
    sc_prop=('prop_SC', 'first')
).reset_index()
print("Group counts:")
print(subject_props['group'].value_counts())
print()
print("SC property distribution:")
print(subject_props['sc_prop'].value_counts())
print()
print("SC property by group:")
print(pd.crosstab(subject_props['group'], subject_props['sc_prop']))

print()
print("=" * 60)
print("SANITY CHECK 6: Trial counts per role per block")
print("=" * 60)
# Verify subjects got adequate exposure to each role
trial_counts = trials.groupby(
    ['subject_id', 'task', 'block', 'trial_role']
)['pressed'].count()
print("Trial count distribution per subject per role per block:")
print(trial_counts.describe().round(1))
print()
print("Minimum trial count per role (should be >5 for reliable press rates):")
print(trial_counts.groupby('trial_role').min())

SANITY CHECK 1: Outcome probabilities vs. design
                     mean  count
trial_role pressed              
OC         0        0.908    599
           1        0.899    991
R          0        0.088   1101
           1        0.089    662
SC         0        0.051   4079
           1        0.952   3768

Expected:
  SC, pressed=1 → ~0.95
  SC, pressed=0 → ~0.05
  OC (any)      → ~0.90
  R  (any)      → ~0.10

SANITY CHECK 2: OC/R property swap between tasks
Subjects with correct OC/R swap: 0/70
                          OC_task1  OC_task2   R_task1   R_task2  \
subject_id                                                         
542bdb6dfdf99b324ea37c3a      size      size    colour    colour   
56153d5e7ffc8a000a811ba8      size      size    colour    colour   
56eeb66a524cc0000a5fa4a2      size      size  velocity  velocity   
5827634ea80bf4000199994b  velocity  velocity    colour    colour   
598f22af33481c000164d359  velocity  velocity    colour    colour   
...             

In [5]:
# ── Fix: Reconstruct correct Task 2 OC and R property labels ──────────────────

# Clean up any duplicate prop_SC columns from previous runs
if 'prop_SC_x' in df_clean.columns:
    df_clean = df_clean.rename(columns={'prop_SC_x': 'prop_SC'})
if 'prop_SC_y' in df_clean.columns:
    df_clean = df_clean.drop(columns=['prop_SC_y'])

# Get each subject's Task 1 role assignments (ground truth)
task1_roles = df_clean[df_clean['task'] == 1].groupby('subject_id')[
    ['prop_SC', 'prop_OC', 'prop_R']
].first().reset_index()
task1_roles.columns = ['subject_id', 'prop_SC_t1', 'prop_OC_t1', 'prop_R_t1']

# Merge onto clean data — use prop_SC_t1 to avoid collision with existing prop_SC
df_clean = df_clean.merge(task1_roles, on='subject_id', how='left')

# Reconstruct correct OC and R labels per task
df_clean['prop_OC'] = df_clean.apply(
    lambda r: r['prop_OC'] if r['task'] == 1 else r['prop_R_t1'], axis=1
)
df_clean['prop_R'] = df_clean.apply(
    lambda r: r['prop_R'] if r['task'] == 1 else r['prop_OC_t1'], axis=1
)

# Drop helper columns
df_clean = df_clean.drop(columns=['prop_SC_t1', 'prop_OC_t1', 'prop_R_t1'])

# Verify fix
verify = df_clean.groupby(['subject_id', 'task'])[['prop_OC', 'prop_R']].first().unstack('task')
verify.columns = ['OC_t1', 'OC_t2', 'R_t1', 'R_t2']
verify['swap_correct'] = (
    (verify['OC_t1'] == verify['R_t2']) &
    (verify['R_t1'] == verify['OC_t2'])
)
n_correct = verify['swap_correct'].sum()
print(f'OC/R swap now correct for: {n_correct}/70 subjects')

# Verify SC unchanged
sc_check = df_clean.groupby(['subject_id', 'task'])['prop_SC'].first().unstack('task')
sc_check.columns = ['SC_t1', 'SC_t2']
print(f'SC property constant across tasks: {(sc_check["SC_t1"] == sc_check["SC_t2"]).all()}')

print(f'\nFinal columns: {df_clean.columns.tolist()}')
print(f'Shape: {df_clean.shape}')

# Resave
df_clean = df_clean.sort_values(['subject_id', 'task', 'block', 'trial_n'])
df_clean.to_csv(
    '/Users/ninaedgley/contingency_simulation/data/processed/clean_data.csv',
    index=False
)
print(f'Clean data resaved: {len(df_clean)} rows, {df_clean["subject_id"].nunique()} subjects')

OC/R swap now correct for: 70/70 subjects
SC property constant across tasks: True

Final columns: ['rt', 'subject_id', 'group', 'phase', 'task', 'block', 'trial_n', 'watch_only', 'prop_SC', 'prop_OC', 'prop_R', 'trial_colour', 'trial_size', 'trial_velocity', 'trial_role', 'pressed', 'outcome', 'outcome_shown', 'completion_ms', 'completion_min']
Shape: (12600, 20)
Clean data resaved: 12600 rows, 70 subjects
